# Beyond spectral indices: 5-fold cross-validation on a GPU

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → **Run all**.

What this runs on CaBuAr:
- Each of the 5 folds is the test fold once, so every chip is tested exactly once.
- Each fold trains the tuned dNBR threshold, both GBDTs and both U-Nets. The U-Nets train on full chips with early stopping.
- The paper's split (test fold 4) is repeated with 2 extra seeds.
- The paper's split also gets the full explanation analysis (IG, GradientSHAP, TreeSHAP, occlusion, randomisation check).

It takes about **2.5–3.5 hours** on a T4. Results are written to Google Drive as they finish. If Colab disconnects, reconnect and **Run all** again: finished steps are skipped. It re-cuts the chips, about 10 minutes, and then continues.

At the end, download **`MyDrive/bsi_cv_results.zip`** and send it back.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUNS = "/content/drive/MyDrive/bsi_cv"

In [ ]:
%cd /content
!test -d repo || git clone -q https://github.com/inayatarshad/beyond-spectral-indices.git repo
%cd /content/repo
!git pull -q
!pip install -q -e .

## Run the cross-validation
This is the long step. The log shows the current fold, model and epoch.

In [ ]:
!python scripts/run_cv.py configs/burn_gpu.yaml --runs-root $RUNS --data-root /content/data

## Package the results
This keeps all CSV/JSON outputs, plus the models of the paper's split (needed for the example figure).

In [ ]:
import zipfile, pathlib
root = pathlib.Path(RUNS)
zpath = pathlib.Path("/content/drive/MyDrive/bsi_cv_results.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in root.rglob("*"):
        if p.is_dir() or "_prepare" in p.parts:
            continue
        if p.suffix in (".pt", ".txt") and not p.is_relative_to(root / "test4_seed0"):
            continue
        z.write(p, p.relative_to(root.parent))
print(f"{zpath}  {zpath.stat().st_size / 1e6:.0f} MB  -> download it from Google Drive")